# Import required libraries. The energies will be computed using PySCF, library for computational chemistry. It only runs on Mac, Linux, and Linux subsystems for Windows (e.g. Ubuntu). 

In [ ]:
from pyscf import gto
import numpy as np
import time

# First, let's see what the potential energy surface for $H_2$ looks like

In [ ]:
# r_list is a list of all of the bond lengths at which we want to obtain the ground state energy
r_list = np.arange(0.7, 4.1, 0.1)
energy_list = [] # This will hold the results

# For each bond length, we will compute the ground state energy
for r in r_list:
    mol = gto.Mole()
    mol.atom = f'''H  0 0 0; H {r} 0 0''' # We create the molecule by specifying the atoms and their coordinates (x,y,z)
    mol.basis = 'sto-3g' # This is the basis set in which the schrodinger equation will be solved
    mol.unit = 'bohr' # The units for the result
    mol.build() # This builds the molecule
    
    # The following code applies density functional theory to calculate the ground state energy
    mf = mol.KS()
    mf.xc = 'b3lyp'
    energy_list.append(mf.kernel())

    # Orbital energies, Mulliken population etc.
    #mf.analyze()

In [ ]:
# Now we plot the potential energy surface

import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))
plt.plot(r_list, energy_list, 'b-*')
plt.xlabel('Bond Length', size=25)
plt.ylabel('Energy',size=25)
plt.title('Potential Energy Surface For $\mathregular{H_2}$',size=25)
plt.xticks(fontsize= 20)
plt.yticks(fontsize= 20)
plt.show()

# Now, we create functions for calculating the energies

In [ ]:
def DFT_H2(r1,r2):
    mol = gto.Mole()
    mol.atom = f'''H  {r1} 0 0; H {r2} 0 0'''
    mol.basis = 'sto-3g'
    mol.unit = 'bohr'
    mol.build()

    mf = mol.KS()
    mf.xc = 'b3lyp'
    energy = mf.kernel()
    
    return energy


def force(r1, r2, dx):
        
    force = -(DFT_H2(r1,r2+dx) - DFT_H2(r1,r2-dx))/(2*dx)
    
    return force

def force2(r1, r2, nucleus, dx):
    
    if nucleus == 1:
        
        force = -(DFT_H2(r1+dx,r2) - DFT_H2(r1-dx,r2))/(2*dx)
    
    if nucleus == 2:
        
        force = -(DFT_H2(r1,r2+dx) - DFT_H2(r1,r2-dx))/(2*dx)
    
    return force

In [ ]:
dx = 0.00001
current_positions = [1.301,2.699] #positions of the particles such that center of mass at 2 a.u.
current_velocities = [0,0] # initial velocity of each particle # This list will hold the new calculated forces
current_forces = [force(current_positions[0], current_positions[1], dx)]*2
print(current_forces)

In [ ]:
dx = 0.00001

bond_length = 2 #initial bond length
position_CM = 2 #initial position of center of mass
#r1 = (2*position_CM-bond_length)/2
#r2 = r1+bond_length
r1 = 0

current_positions = [r1,r1+bond_length] # initial positions of the particles
current_velocities = [0,0] # initial velocity of each particle # This list will hold the new calculated forces
current_forces = [force2(current_positions[0], current_positions[1], 1, dx), force2(current_positions[0], current_positions[1], 2, dx)]
mass = [1.0078,1.0078] # mass of each particle

dt = 0.1
simulation_length = 20
positions_list = [current_positions.copy()]
velocities_list = [current_velocities.copy()]
forces_list = [current_forces.copy()]
bondLength_list = [bond_length]
vel_diff = [0]
tic = time.perf_counter()

for i in np.arange(0, simulation_length, dt):
    
    for j in range(len(current_positions)):
        current_positions[j] = current_positions[j] + current_velocities[j]*dt + (current_forces[j]*dt**2)/(2*mass[j])
    
    new_forces = [0,0]
    for j in range(len(new_forces)):
        new_forces[j] = force2(current_positions[0], current_positions[1], j+1, dx)
        
    for j in range(len(current_velocities)):
        current_velocities[j] = current_velocities[j] + ((new_forces[j] + current_forces[j])*dt)/(2*mass[j])
        
    for j in range(len(current_forces)):
        current_forces[j] = new_forces[j]
    
    bondLength_list.append(current_positions[1] - current_positions[0])
    vel_diff.append(current_velocities[1] - current_velocities[0])
    positions_list.append(current_positions.copy())
    velocities_list.append(current_velocities.copy())
    forces_list.append(current_forces.copy())

toc = time.perf_counter()
print(f"Completed simulation in {toc - tic:0.4f} seconds")

In [ ]:
print(bondLength_list)

In [ ]:
#print(positions_list)
#print(velocities_list)
#print(forces_list)

positionCM = []
for i in positions_list:
    positionCM.append((i[0]+i[1])/2)
    
velocityCM = []
for i in velocities_list:
    velocityCM.append((i[0]+i[1])/2)
    
print(positionCM)
print(velocityCM)

In [ ]:
position1 = []
for i in positions_list:
    position1.append(i[0])
    
velocity1 = []
for i in velocities_list:
    velocity1.append(i[0])
    
print(position1,'\n')
print(velocity1,'\n\n')

position2 = []
for i in positions_list:
    position2.append(i[1])
    
velocity2 = []
for i in velocities_list:
    velocity2.append(i[1])
    
print(position2,'\n')
print(velocity2,'\n')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))
plt.plot(bondLength_list, vel_diff, 'b-*')
plt.plot(position1, velocity1, 'r-*')
plt.plot(position2, velocity2, 'g-*')
plt.xlabel('Position', size=25)
plt.ylabel('Velocity',size=25)
plt.title('Phase Space For $\mathregular{H_2}$',size=25)
plt.xticks(fontsize= 20)
plt.yticks(fontsize= 20)
plt.show()

In [ ]:
from matplotlib import pyplot as plt
from celluloid import Camera

    
fig = plt.figure(10,10)
plt.xlim([-2, 2])
plt.ylim([-1,1])
camera = Camera(fig)
for i in range(len(positions_list)):
    plt.scatter(positions_list[i],[0]*len(current_positions), s=120, c=[1]*len(current_positions))
    plt.scatter(positionCM[i], 0, s=120, c=[2]*len(current_positions))
    text = "T = " + str(dt*i)
    plt.text(1, 0.5, text, ha='left',wrap=False)
    camera.snap()

animation = camera.animate()
animation.save('H2_gif.gif', writer = 'pillow', fps=40)

In [ ]:
print(positions_list)